## Loading the Model

In [1]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from huggingface_hub import hf_hub_download, notebook_login
import numpy as np
import einops
import textwrap
from typing import Literal
import plotly.express as px
from functools import partial
import dataclasses
from IPython.display import display, HTML
import gc
import pandas as pd
from safetensors.torch import load_file
import torch
import torch.nn as nn

In [2]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')

login(token = HF_TOKEN)

In [3]:
import pandas as pd

ecqa_data = pd.read_parquet("/content/validation-00000-of-00001.parquet")

In [4]:
ecqa_data

,Unnamed: 0,q_no,q_concept,q_text,q_op1,q_op2,q_op3,q_op4,q_op5,q_ans,taskA_pos,taskA_neg,taskB
0,0,af836abc58e0daf36df1d8d6830b70c5,applying for job,What does someone typically feel when applying...,horror,anxiety and fear,rejection,increased workload,being employed,anxiety and fear,Anxiety is the feeling of fear when doing some...,Horror is an intense feeling of fear or shock....,Anxiety is the feeling of fear when doing some...
1,1,2ac72eaf30a633c410b1bd658bbef0ba,car,Where do cars usually travel at very high speeds?,freeway,road,race track,alley,parking lot,race track,Racers compete on the race track\nCompeting in...,Racers do not compete on the freeway\nRacers d...,Racers compete on the race track and not on th...
2,2,6bb2a47c94ef3f23c3fb72079015dd82,take oath,"Each witness had to take oath, this oath was t...",laugh,truthful,not lie,think,sense of duty,not lie,A person should not lie after taking oath.\nEa...,The oath was not to laugh.\nEach witness had t...,A person should not lie after taking oath. Eac...
3,3,3b6a8321ebfba3fa2f054345190e4205,dining table,Where do people eat at dining tables together ...,house,formal dining room,cafeteria,conference room,doing jigsaw puzzles on,cafeteria,Cafeteria is a restaurant in which customers s...,House is to go to the place where one lives pe...,Cafeteria is a restaurant in which customers s...
4,4,6b0bf501aa68b06ddc5ad72ac5ff68fc,mouse,"Though a mouse might prefer your house, you mi...",tin,department store,garden,small hole,cupboard,garden,Mouse can be found in open areas near houses l...,Tin is a box\nDeaprtment store is tyep of stor...,"If a mouse is not in house, he is liekly to be..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1085,1085,dc7cc405118f2307b53baf038637384a,play cards,What is illegal to do when you play cards at a...,remember,help,count,winning,dealing,count,"At casino, when people play, they cannot count...",Rememebring is not related to a illegal task a...,"When people play cards at casino, they play fo..."
1086,1086,7d2b4837d2fc48881f906ba5adc47801,eating,What is eating an unhealthy meal likely to cause?,gas,gaining weight,electrical circuit,indigestion,getting full,gas,Uhealthy meal creates gas in stomach as they u...,eating unhealthy meal daily cause gaining weig...,Uhealthy meal creates gas in stomach as they u...
1087,1087,1daf2f65747cc0e5840530cdc0ea991d,fitting room,What popular clothing retailer often has a fit...,gap,car dealership,department store,mall,appropriate,gap,Gap is famous clothing retailer.\nGap has fitt...,.\nCar dealership is not clothing retailer.\nD...,Gap is famous clothing retailer and Gap has f...
1088,1088,91f992d260abe43d5002d8231c2f5b20,round brush,If you brush your hair while bathing what is a...,kitchen,shower,department store,art supplies,hair salon,shower,Shower is the place of bathing.\nWe keep a rou...,"A person baths in shower, so kitchen is not us...",Shower is the place of bathing. We keep a roun...


In [5]:
ecqa_data.iloc[0]["q_text"]

'What does someone typically feel when applying for a job?'

In [6]:
ecqa_data.iloc[0]["q_text"]

'What does someone typically feel when applying for a job?'

In [7]:
def format_prompt(df, row_idx):
    row = df.iloc[row_idx]

    # Extract question and options
    question = row['q_text']
    options = [
        row['q_op1'],
        row['q_op2'],
        row['q_op3'],
        row['q_op4'],
        row['q_op5']
    ]

    # Format options as a), b), c), etc.
    options_str = " ".join([f"{chr(97+i)}) {opt}" for i, opt in enumerate(options)])

    # Create the prompt
    prompt = f"""<start_of_turn>user Question: {question}
    Options: {options_str}

    Before selecting an answer, explain your reasoning process. Explain the reasoning behind every option.

    Reasoning: [Your step-by-step thinking]
    Answer: [Your final choice] <end_of_turn>
    <start_of_turn>model"""

    return prompt

In [8]:
formatted_prompt = format_prompt(ecqa_data, 0)
print(formatted_prompt)

<start_of_turn>user Question: What does someone typically feel when applying for a job?
    Options: a) horror b) anxiety and fear c) rejection d) increased workload e) being employed

    Before selecting an answer, explain your reasoning process. Explain the reasoning behind every option.

    Reasoning: [Your step-by-step thinking]
    Answer: [Your final choice] <end_of_turn>
    <start_of_turn>model


In [9]:
gemma_version = "google/gemma-3-4b-it"
# corresponding_sae_folder = "google/gemma-scope-2-1b-it"

model = AutoModelForCausalLM.from_pretrained(
    gemma_version,
    device_map='auto',
)
tokenizer =  AutoTokenizer.from_pretrained(gemma_version)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [10]:
model.eval()

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

## Sparse Autoencoders

OK, so we have got our Gemma model loaded, and we can sample from it to get sensible stuff. Now, let's load one of our SAEs.

GemmaScope actually contains over four hundred SAEs, but for now we'll just load one on the residual stream at the end of layer 20 (of 26, note that layers start at 0 so this is the 21st layer. This is a fairly late layer, so the model should have time to find more abstract concepts!).

Note, we're loading from the `resid_post` directory here. We can also load from `resid_post_all` to get SAEs trained on every single layer (not just a subset of 4 layers), but a smaller range of widths and L0 values.

<details><summary>What is the residual stream?</summary>

Transformers have skip connections, which means that the output of each block is the output of each sublayer *plus* the input to the block. This means that each sublayer (attention or MLP) actually only has a fairly small effect on the output of the block, since most of it comes from all the earlier layers. We call the output of a block (including skip connections) the **residual stream**.

Everything communicated from earlier layers to later layers must go via the residual stream, so it acts as a "bottleneck" in the transformer, essentially capturing everything the model has "thought" so far. This means it is often a natural thing to study, since it will contain everything important going on in the model.
</details>


In [10]:
# SAE config
LAYER = 29  # options are {7, 13, 17, 22}
WIDTH = "262k"   # options are {16k, 65k, 262k, 1m}
L0 = "small"  # options are {small, medium, big}

corresponding_sae_folder = "google/gemma-scope-2-4b-it"
path_to_params = hf_hub_download(
    repo_id=corresponding_sae_folder,
    filename=f"resid_post_all/layer_{LAYER}_width_{WIDTH}_l0_{L0}/params.safetensors",
)

params = load_file(path_to_params)

Our SAEs are **JumpReLU** SAEs, meaning they are a standard 2-layer neural network with a JumpReLU activation function (a ReLU with a discontinuous jump).

<!-- The mapping from input to hidden activations is defined by the weight and bias parameters `W_enc` and `b_enc`, and the mapping from hidden activations back to reconstructed input is defined by `W_dec` and `b_dec`. The `threshold` parameter determines the size of the discontinuity for JumpReLU. -->

### Implementing the SAE


We now define the forward pass of the SAE for pedagogical purposes (in practice, we recommend using the implementation in SAELens).

We have 5 important parameters below:

- `w_enc`, the encoder matrix (which maps from inputs to pre-activation latent values)
- `b_enc`, the bias added onto these pre-activation latent values
- `threshold`, which determines how we apply our JumpReLU activation function
- `w_dec`, the decoder matrix (which maps from post-ReLU latent values to reconstructed activations)
- `b_dec`, the bias which is added to the final reconstruction

You can ignore `affine_skip_connection` for now; we'll come back to it in the "transcoders" section.

In [13]:
class JumpReLUSAE(nn.Module):
  def __init__(self, d_in, d_sae, affine_skip_connection=False):
    # Note that we initialise these to zeros because we're loading in pre-trained weights.
    # If you want to train your own SAEs then we recommend using blah
    super().__init__()
    self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
    self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
    self.threshold = nn.Parameter(torch.zeros(d_sae))
    self.b_enc = nn.Parameter(torch.zeros(d_sae))
    self.b_dec = nn.Parameter(torch.zeros(d_in))
    if affine_skip_connection:
      self.affine_skip_connection = nn.Parameter(torch.zeros(d_in, d_in))
    else:
      self.affine_skip_connection = None

  def encode(self, input_acts):
    pre_acts = input_acts @ self.w_enc + self.b_enc
    mask = (pre_acts > self.threshold)
    acts = mask * torch.nn.functional.relu(pre_acts)
    return acts

  def decode(self, acts):
    return acts @ self.w_dec + self.b_dec

  def forward(self, x):
    acts = self.encode(x)
    recon = self.decode(acts)
    if self.affine_skip_connection is not None:
      return recon + x @ self.affine_skip_connection
    return recon

In [14]:
d_model, d_sae = params["w_enc"].shape
sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae.cuda()

JumpReLUSAE()

In [14]:
d_model

2560

### Running the SAE on model activations


Let's first get out some activations from the model at the SAE target site. We'll demonstrate how to do this 'manually' first, by using Pytorch hooks. Note that this is not particularly good practice, and it's probably more practical to use a library like TransformerLens to handle hooking the SAE into a model forward pass. But for illustrative purposes, it's useful to see how it's done.

We can gather activations at a site by registering a hook. To keep this local, we can wrap this in a function that registers a hook, runs the model, saving the intermediate activation, then removes the hook. (This is basically what TransformerLens is doing under the hood)

In [15]:
def gather_acts_hook(mod, inputs, outputs, cache: dict, key: str, use_input: bool):
  """Generic hook function whic stores activations (either input or output of a particular PyTorch module)."""
  acts = inputs[0].squeeze(0) if use_input else outputs[0]  # inputs usually have a batch dim
  cache[key] = acts
  return outputs


def gather_residual_activations(model, target_layer, inputs):

  cache = {}

  # Add a hook function to store the output of this layer of the model
  handle = model.model.layers[target_layer].register_forward_hook(
      partial(gather_acts_hook, cache=cache, key="resid_post", use_input=False)
  )

  # Forward pass inside a try/except/finally block (useful just in case our hook breaks
  # and we can't remove it!)
  try:
    _ = model.forward(inputs)
  finally:
    handle.remove()

  return cache["resid_post"]

Now, we can run our SAE on the saved activations.

In [16]:
def fwd_pass_with_sae_intervention(model, sae, target_layer, inputs):
  # Forward pass to get logits & hidden activations
  model_output_clean = model.forward(inputs, output_hidden_states=True)
  logits_clean = model_output_clean.logits[0]  # (seq, d_vocab)
  input_acts = model_output_clean.hidden_states[target_layer + 1][0]  # (seq, d_model)

  # Get the SAE reconstruction
  recon = sae.forward(input_acts.to(torch.float32))

  def intervene_on_target_act_hook(mod, inputs, outputs):
    outputs[0][1:] = recon[1:]
    return outputs

  handle = model.model.layers[target_layer].register_forward_hook(intervene_on_target_act_hook)
  try:
    model_output = model.forward(inputs)
  finally:
    handle.remove()

  # Get logits from this corrupted forward pass
  logits = model_output.logits[0]

  return logits_clean, logits


def cross_entropy_loss(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
  """Measures avg cross entropy loss."""
  logprobs = logits[:-1].log_softmax(dim=-1)
  tokens = tokens[1:]
  correct_logprobs = logprobs[torch.arange(len(tokens)), tokens]
  return -correct_logprobs

In [17]:
formatted_prompt = format_prompt(ecqa_data, 50)
# print(formatted_prompt)
inputs = tokenizer.encode(formatted_prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=500)
print(tokenizer.decode(outputs[0]))

<bos><start_of_turn>user Question: My cat really dislikes many things, what does he dislike the most?
    Options: a) eat vegetables b) litterbox c) chased by dog d) washed e) bathed

    Before selecting an answer, explain your reasoning process. Explain the reasoning behind every option.

    Reasoning: [Your step-by-step thinking]
    Answer: [Your final choice] <end_of_turn>
    <start_of_turn>model
Okay, let’s break down this question and figure out the most likely answer. We need to consider what cats generally dislike and which of the options would be most distressing to them.

**Reasoning:**

* **a) eat vegetables:** Most cats are obligate carnivores, meaning their bodies are designed to thrive on meat. While some cats will nibble on veggies out of curiosity, actively *forcing* them to eat vegetables is unlikely to be their biggest aversion. It’s a minor annoyance, not a deeply disliked experience.

* **b) litterbox:** Litterbox issues are extremely common in cats. Many cats ar

In [18]:
def get_top_k_features_from_sae(sample_input, k):

    inputs = tokenizer.encode(sample_input, return_tensors="pt", add_special_tokens=True).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=500)
    target_act = gather_residual_activations(model, LAYER, inputs)
    sae_acts = sae.encode(target_act.to(torch.float32))
    recon = sae.decode(sae_acts)

    top_acts, top_latents = sae_acts.squeeze().mean(0).topk(k)
    print("Top features across FULL generation (prompt):")
    print("-" * 50)

    for act, idx in zip(top_acts, top_latents):
      print(f"{act:>6.1f} | {idx}")

    return top_acts, top_latents, sae_acts

In [19]:
formatted_prompt = format_prompt(ecqa_data, 10)
# print(formatted_prompt)

top_acts, top_latents, sae_acts = get_top_k_features_from_sae(formatted_prompt, 10)

AttributeError: 'Gemma3Model' object has no attribute 'layers'

In [ ]:
top_latents

In [ ]:
feature_idx = 6196

str_toks = tokenizer.tokenize(formatted_prompt, add_special_tokens=True)
activations = sae_acts.squeeze(0)[:, feature_idx].tolist()

def html_activations(str_toks: list[str], activations: list[float]):
  return "".join(
      f'<span style="background-color: rgba(255,0,0,{v}); padding: 4px 0px;">{t}</span>'
      for t, v in zip(str_toks, np.array(activations) / (1e-6 + np.max(activations)), strict=True)
  )

display(HTML(html_activations(str_toks, activations)))

One guess we might have is that this latent fires on concepts related to science or scientific laws. Let's test this out with a few examples:

In [34]:
for prompt in [
    "Flowers 2 is a model release from Google DeepMind",
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit",
    "Gravity describes how massive objects attract one another",
    "A charge accelerating through an electric field experiences a force",
    "Chemical fuel stores energy in molecular bonds, which is released"
]:
  inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")
  _target_acts = gather_residual_activations(model, LAYER, inputs)

  _sae_acts = sae.encode(_target_acts.to(torch.float32))

  str_toks = tokenizer.tokenize(prompt, add_special_tokens=True)
  display(HTML(html_activations(str_toks, _sae_acts[:, feature_idx].tolist())))
  print()

Okay, so it doesn't fire on the gravity sentence, but it does fire on both the other physics-related sentences as soon as they start talking about forces, energies or fields. This gives us a more specific idea of the concepts this latent might represent.

<!-- This theory seems reasonable: it fires on both the sentences related to physical forces (note that it doesn't seem to just be a "gravity" latent given how it fires on the second of these two sentences). -->

<!-- Okay, so it seems like this might be the case, although the activation is more consistent when describing gravity than on other forces. -->

We can investigate this further by looking at the latent's unembedding, in other words **what words get predicted strongest when this latent fires.** From the results below, we might guess this latent represents a more specific concept: entropy and thermodynamics.

In [28]:
feature_idx = 5149
w_u = model.lm_head.weight  # shape (d_vocab, d_model)
w_u_eff = w_u * model.model.language_model.norm.weight

decoder_vector = sae.w_dec[feature_idx]  # shape (d_model,)

decoder_vector = decoder_vector.to(w_u_eff.dtype)

top_activations, top_tokens = torch.topk(w_u_eff @ decoder_vector, k=10)

for act, tok in zip(top_activations, top_tokens):
    print(f"{act:.4f} | {tokenizer.decode(tok)}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.25 GiB. GPU 0 has a total capacity of 14.74 GiB of which 538.12 MiB is free. Process 69977 has 14.21 GiB memory in use. Of the allocated memory 13.81 GiB is allocated by PyTorch, and 280.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

<!-- This definitely increases credence in our theory that this feature specifically relates to gravity! -->

Lastly, we can try **steering with this feature**. This means intervening in the residual stream of the model to add some multiple of this feature's decoder vector, so that we can change the behaviour of the model during generation.

You should see that when we steer the model on this "physical force feature", it starts talking more about enetry (specifically entropy or thermodynamics). Note that steering can often be fragile; it's difficult to choose the intervention layer and steering coefficient in a way that gives the expected behavioural change without also breaking the model's coherence. If you're curious, you can try increasing the `coeff` parameter below and seeing what happens!

In [ ]:
def generate_with_steering(model, sae, inputs, target_layer, feature_idx: int, coeff: float):

  def steering_hook(mod, inputs, outputs):
    output = outputs[0]

    if output.shape[1] == 1:
        # Cached: [batch, d_model] where batch dimension is 1
        avg_norm = torch.norm(output, dim=-1, keepdim=True)
        output += coeff * avg_norm * sae.w_dec[feature_idx]
    else:
        # First pass: [seq_len, d_model]
        avg_norm = torch.norm(output[-1:], dim=-1, keepdim=True)
        steering = coeff * avg_norm * sae.w_dec[feature_idx]
        output[-1:] += steering

    return outputs


  handle = model.model.layers[target_layer].register_forward_hook(steering_hook)
  try:
    outputs = model.generate(input_ids=inputs, max_new_tokens=500, do_sample=False)
    output_str = tokenizer.decode(outputs[0])
  finally:
    handle.remove()

  return output_str.split("<start_of_turn>model")[1].strip()


formatted_prompt = format_prompt(ecqa_data, 1)
inputs = tokenizer.encode(formatted_prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

print(formatted_prompt)
print("======================= NO STEERING =======================")
output_str = generate_with_steering(
    model=model,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.0,
)
print(textwrap.fill(output_str))
print("======================= STEERING =======================")
output_str_steered = generate_with_steering(
    model=model,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.14,
)
print(textwrap.fill(output_str_steered))

In [ ]:
def generate_with_steering(model, sae, inputs, target_layer, feature_idx: int, coeff: float):

  def steering_hook(mod, inputs, outputs):
    output = outputs
    # We have to be careful about KV caching! This logic handles different cases depending on
    # whether this is the first forward pass or a cached pass.
    if output.shape[1] == 1:
      avg_norm = torch.norm(output, dim=-1)
      output += coeff * avg_norm * sae.w_dec[feature_idx]
    else:
      # avg_norm = torch.norm(output[0, 1:], dim=-1, keepdim=True)
      # output[0, 1:] += coeff * avg_norm * sae.w_dec[feature_idx]
      avg_norm = torch.norm(output[0, -1:], dim=-1, keepdim=True)
      output[0, -1:] += coeff * avg_norm * sae.w_dec[feature_idx]

    return outputs

  handle = model.model.language_model.layers[target_layer].register_forward_hook(steering_hook)
  try:
    outputs = model.generate(input_ids=inputs, max_new_tokens=80, do_sample=False)
    output_str = tokenizer.decode(outputs[0])
  finally:
    handle.remove()

  return output_str.split("<start_of_turn>model")[1].strip()

def format_prompt(user_prompt: str) -> str:
  return f"""<start_of_turn>user
{user_prompt}<end_of_turn>
<start_of_turn>model
"""

user_prompt = "The chickens are my favourite animals."
inputs = tokenizer.encode(format_prompt(user_prompt), return_tensors="pt", add_special_tokens=True).to("cuda")

print(user_prompt)
print("======================= NO STEERING =======================")
output_str = generate_with_steering(
    model=model,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.0,
)
print(textwrap.fill(output_str))
print("======================= STEERING =======================")
output_str_steered = generate_with_steering(
    model=model,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.07,
)
print(textwrap.fill(output_str_steered))

In [16]:
def get_features_with_full_generation(model, tokenizer, sae, sample_input, layer_idx, k=10, max_new_tokens=200):
    """
    Generate text and capture SAE activations for ALL tokens (prompt + generated).

    Returns:
        top_acts: Top k activation values
        top_latents: Top k feature indices
        sae_acts: Full SAE activations [total_seq_len, num_features]
        token_ids: Token IDs of complete sequence
        generated_text: The full generated text
    """

    # Storage for all hidden states during generation
    all_hidden_states = []

    def capture_hook(module, input, output):
        """Hook that captures hidden states at each generation step"""
        # output[0] shape: [batch, seq_len, d_model]
        all_hidden_states.append(output[0].detach().cpu())
        return output

    # Register hook on target layer
    target_layer = model.model.layers[layer_idx]
    handle = target_layer.register_forward_hook(capture_hook)

    try:
        # Tokenize input
        inputs = tokenizer.encode(sample_input, return_tensors="pt", add_special_tokens=True).to("cuda")

        # Create attention mask to fix the warning
        attention_mask = torch.ones_like(inputs)

        # Generate with hook active
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Greedy for consistency
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode full output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

    finally:
        # Always remove hook
        handle.remove()

    # First forward pass contains all prompt tokens: [1, prompt_len, d_model]
    full_hidden = all_hidden_states[0].squeeze(0)  # [prompt_len, d_model]

    # Each subsequent pass contains one generated token: [1, 1, d_model]
    for hidden in all_hidden_states[1:]:
        # Ensure consistent shape before concatenating
        hidden_squeezed = hidden.squeeze(0)  # Remove batch dim

        # Handle both [1, d_model] and [d_model] cases
        if hidden_squeezed.dim() == 1:
            hidden_squeezed = hidden_squeezed.unsqueeze(0)  # Make it [1, d_model]

        full_hidden = torch.cat([full_hidden, hidden_squeezed], dim=0)

    # Now full_hidden has shape: [prompt_len + num_generated, d_model]


    full_hidden = full_hidden.to("cuda")
    sae_acts = sae.encode(full_hidden.to(torch.float32))  # [total_len, num_features]

    # Get top k features (averaged across entire sequence)
    top_acts, top_latents = sae_acts.mean(0).topk(k)

    print("Top features across FULL generation (prompt + output):")
    print("-" * 50)
    for act, idx in zip(top_acts, top_latents):
        print(f"{act:>6.1f} | Feature {idx}")

    return top_acts, top_latents, sae_acts, outputs[0], generated_text

def visualize_feature_on_full_generation(tokenizer, token_ids, sae_acts, feature_idx):
    """
    Visualize a feature's activations across the entire generation.
    Red intensity shows activation strength.
    Preserves original text formatting (newlines, spacing).
    """

    # Decode the full text to preserve formatting
    full_text = tokenizer.decode(token_ids, skip_special_tokens=False)

    # Decode each token individually to get token boundaries
    str_tokens = [tokenizer.decode([tok_id]) for tok_id in token_ids]

    # Get activations for this feature
    activations = sae_acts[:, feature_idx].cpu().detach().numpy()

    # Normalize to 0-1 for color intensity
    normalized = activations / (1e-6 + np.max(activations))

    # Create HTML with colored spans, preserving original text structure
    html_parts = []
    for token, intensity in zip(str_tokens, normalized):
        # Escape HTML special characters but preserve newlines
        token_display = (token
                        .replace('&', '&amp;')
                        .replace('<', '&lt;')
                        .replace('>', '&gt;')
                        .replace('\n', '<br>')  # Preserve line breaks
                        .replace(' ', '&nbsp;'))  # Preserve spaces

        html_parts.append(
            f'<span style="background-color: rgba(255,0,0,{intensity}); '
            f'color: black; padding: 2px 0px; margin: 0;">{token_display}</span>'
        )

    html_output = ''.join(html_parts)

    # Display with pre-formatted text styling
    display(HTML(f'''
        <div style="font-family: 'Courier New', monospace;
                    font-size: 13px;
                    padding: 15px;
                    background: white;
                    border: 1px solid #ddd;
                    border-radius: 5px;
                    overflow-x: auto;
                    line-height: 1.6;">
            {html_output}
        </div>
    '''))

In [24]:
def get_features_with_full_generation(model, tokenizer, sae, sample_input, layer_idx, k=10, max_new_tokens=200):
    """
    Generate text and capture SAE activations for ALL tokens (prompt + generated).

    Returns:
        top_acts: Top k activation values
        top_latents: Top k feature indices
        sae_acts: Full SAE activations [total_seq_len, num_features]
        token_ids: Token IDs of complete sequence
        generated_text: The full generated text
    """

    # Storage for all hidden states during generation
    all_hidden_states = []

    def capture_hook(module, input, output):
        """Hook that captures hidden states at each generation step"""
        # output[0] shape: [batch, seq_len, d_model]
        all_hidden_states.append(output[0].detach().cpu())
        return output

    # For Gemma3ForConditionalGeneration with language_model
    target_layer = model.model.language_model.layers[layer_idx]
    handle = target_layer.register_forward_hook(capture_hook)

    try:
        # Tokenize input
        inputs = tokenizer.encode(sample_input, return_tensors="pt", add_special_tokens=True).to("cuda")

        # Create attention mask to fix the warning
        attention_mask = torch.ones_like(inputs)

        # Generate with hook active
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Greedy for consistency
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode full output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

    finally:
        # Always remove hook
        handle.remove()

    # First forward pass contains all prompt tokens: [1, prompt_len, d_model]
    full_hidden = all_hidden_states[0].squeeze(0)  # [prompt_len, d_model]

    # Each subsequent pass contains one generated token: [1, 1, d_model]
    for hidden in all_hidden_states[1:]:
        # Ensure consistent shape before concatenating
        hidden_squeezed = hidden.squeeze(0)  # Remove batch dim

        # Handle both [1, d_model] and [d_model] cases
        if hidden_squeezed.dim() == 1:
            hidden_squeezed = hidden_squeezed.unsqueeze(0)  # Make it [1, d_model]

        full_hidden = torch.cat([full_hidden, hidden_squeezed], dim=0)

    # Now full_hidden has shape: [prompt_len + num_generated, d_model]

    full_hidden = full_hidden.to("cuda")
    sae_acts = sae.encode(full_hidden.to(torch.float32))  # [total_len, num_features]

    # Get top k features (averaged across entire sequence)
    top_acts, top_latents = sae_acts.mean(0).topk(k)

    print("Top features across FULL generation (prompt + output):")
    print("-" * 50)
    for act, idx in zip(top_acts, top_latents):
        print(f"{act:>6.1f} | Feature {idx}")

    return top_acts, top_latents, sae_acts, outputs[0], generated_text

In [25]:
formatted_prompt = format_prompt(ecqa_data, 15)

print("Input prompt:")
print("=" * 80)
print(formatted_prompt)
print("=" * 80)

LAYER = 29  # Adjust to your target layer
top_acts, top_latents, sae_acts, token_ids, generated_text = get_features_with_full_generation(
    model=model,  # or model_it
    tokenizer=tokenizer,
    sae=sae,
    sample_input=formatted_prompt,
    layer_idx=LAYER,
    k=10,
    max_new_tokens=500
)

print("\n" + "=" * 80)
print("FULL GENERATED OUTPUT:")
print("=" * 80)
print(generated_text)
print("=" * 80)

# Visualize the top feature across the full generation
print(f"\nVisualizing Feature {top_latents[0].item()} across ENTIRE generation:")
print("(Darker red = higher activation)\n")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input prompt:
<start_of_turn>user Question: As the fox ran into the forest it disappeared into the what?
    Options: a) nantucket b) barn c) northern hemisphere d) hen house e) undergrowth

    Before selecting an answer, explain your reasoning process. Explain the reasoning behind every option.

    Reasoning: [Your step-by-step thinking]
    Answer: [Your final choice] <end_of_turn>
    <start_of_turn>model
Top features across FULL generation (prompt + output):
--------------------------------------------------
 702.7 | Feature 5149
 686.8 | Feature 10184
 390.0 | Feature 20522
 352.6 | Feature 3345
 332.8 | Feature 1400
 320.9 | Feature 6216
 300.5 | Feature 160308
 276.2 | Feature 8178
 273.6 | Feature 2446
 265.0 | Feature 11560

FULL GENERATED OUTPUT:
<bos><start_of_turn>user Question: As the fox ran into the forest it disappeared into the what?
    Options: a) nantucket b) barn c) northern hemisphere d) hen house e) undergrowth

    Before selecting an answer, explain your reas

In [26]:
visualize_feature_on_full_generation(
    tokenizer=tokenizer,
    token_ids=token_ids,
    sae_acts=sae_acts,
    feature_idx=top_latents[0].item()
)

In [11]:
import time
from typing import List, Dict, Optional
from dataclasses import dataclass
import requests
import numpy as np

@dataclass
class NeuronpediaFeature:
    """Container for feature information from Neuronpedia."""
    feature_idx: int
    description: Optional[str] = None
    frac_nonzero: Optional[float] = None  # Activation density
    max_act_approx: Optional[float] = None  # Max activation value
    max_activating_examples: Optional[List[Dict]] = None
    error: Optional[str] = None


class NeuronpediaClient:
    """Client for interacting with the Neuronpedia API."""

    BASE_URL = "https://www.neuronpedia.org/api"

    def __init__(self, model_id: str, sae_id: str):
        """
        Initialize the Neuronpedia client.

        Args:
            model_id: Model identifier (e.g., 'gemma-3-4b-it')
            sae_id: SAE identifier (e.g., '22-gemmascope-2-mlp-262k')
        """
        self.model_id = model_id
        self.sae_id = sae_id

    def get_feature(self, feature_idx: int) -> NeuronpediaFeature:
        """Fetch feature information from Neuronpedia.

        API endpoint: GET /api/feature/{modelId}/{layer}/{index}
        """
        url = f"{self.BASE_URL}/feature/{self.model_id}/{self.sae_id}/{feature_idx}"

        try:
            response = requests.get(url, timeout=10)

            if response.status_code == 404:
                return NeuronpediaFeature(
                    feature_idx=feature_idx,
                    error="Feature not found on Neuronpedia"
                )

            response.raise_for_status()
            data = response.json()

            # Extract description from explanations if available
            description = None
            if 'explanations' in data and data['explanations']:
                description = data['explanations'][0].get('description', None)

            # Extract activation density (frac_nonzero)
            frac_nonzero = data.get('frac_nonzero', None)

            # Extract max activation
            max_act_approx = data.get('maxActApprox', None)

            # Extract max activating examples
            max_examples = None
            if 'activations' in data:
                max_examples = data['activations']

            return NeuronpediaFeature(
                feature_idx=feature_idx,
                description=description,
                frac_nonzero=frac_nonzero,
                max_act_approx=max_act_approx,
                max_activating_examples=max_examples
            )

        except requests.exceptions.RequestException as e:
            return NeuronpediaFeature(
                feature_idx=feature_idx,
                error=f"API request failed: {str(e)}"
            )

    def get_dashboard_url(self, feature_idx: int) -> str:
        """Get the Neuronpedia dashboard URL for a feature."""
        return f"https://neuronpedia.org/{self.model_id}/{self.sae_id}/{feature_idx}"


def compute_rarity_from_density(frac_nonzero: float, smooth: bool = True) -> float:
    """Compute rarity score from Neuronpedia's frac_nonzero (activation density).

    Args:
        frac_nonzero: Fraction of examples where feature is active (from Neuronpedia)
        smooth: Whether to use smoothing to avoid extreme values

    Returns:
        Rarity score (higher = more distinctive/rare)
    """
    if frac_nonzero is None or frac_nonzero <= 0:
        return 1.0  # Default for missing data

    if smooth:
        # Smoothed rarity: log(1 / (frac + epsilon)) + 1
        return np.log(1.0 / (frac_nonzero + 1e-6)) + 1
    else:
        return np.log(1.0 / frac_nonzero)


def get_features_with_full_generation(
    model,
    tokenizer,
    sae,
    sample_input,
    layer_idx,
    k=10,
    max_new_tokens=200,
    # TF-IDF filtering parameters
    use_tfidf_filtering=True,
    np_client=None,
    max_density=0.005,  # Filter out features with >0.5% density
    rarity_power=2.0,   # Higher = more weight on rarity
    api_delay=0.1       # Delay between API calls
):
    """
    Generate text and capture SAE activations for ALL tokens (prompt + generated).

    Args:
        model: The language model
        tokenizer: The tokenizer
        sae: The SAE model
        sample_input: Input text
        layer_idx: Which layer to hook
        k: Number of top features to return
        max_new_tokens: Max tokens to generate
        use_tfidf_filtering: Whether to use TF-IDF (rarity-weighted) filtering
        np_client: NeuronpediaClient instance (required if use_tfidf_filtering=True)
        max_density: Maximum allowed frac_nonzero (filter out common features)
        rarity_power: Power for rarity weighting (higher = more weight on rarity)
        api_delay: Delay between API calls to avoid rate limiting

    Returns:
        top_acts: Top k activation values
        top_latents: Top k feature indices
        sae_acts: Full SAE activations [total_seq_len, num_features]
        token_ids: Token IDs of complete sequence
        generated_text: The full generated text
        features_info: (Optional) List of dicts with feature metadata if using TF-IDF
    """

    # Storage for all hidden states during generation
    all_hidden_states = []

    def capture_hook(module, input, output):
        """Hook that captures hidden states at each generation step"""
        # output[0] shape: [batch, seq_len, d_model]
        all_hidden_states.append(output[0].detach().cpu())
        return output

    # For Gemma3ForConditionalGeneration with language_model
    target_layer = model.model.language_model.layers[layer_idx]
    handle = target_layer.register_forward_hook(capture_hook)

    try:
        # Tokenize input
        inputs = tokenizer.encode(sample_input, return_tensors="pt", add_special_tokens=True).to("cuda")

        # Create attention mask to fix the warning
        attention_mask = torch.ones_like(inputs)

        # Generate with hook active
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Greedy for consistency
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode full output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

    finally:
        # Always remove hook
        handle.remove()

    # First forward pass contains all prompt tokens: [1, prompt_len, d_model]
    full_hidden = all_hidden_states[0].squeeze(0)  # [prompt_len, d_model]

    # Each subsequent pass contains one generated token: [1, 1, d_model]
    for hidden in all_hidden_states[1:]:
        # Ensure consistent shape before concatenating
        hidden_squeezed = hidden.squeeze(0)  # Remove batch dim

        # Handle both [1, d_model] and [d_model] cases
        if hidden_squeezed.dim() == 1:
            hidden_squeezed = hidden_squeezed.unsqueeze(0)  # Make it [1, d_model]

        full_hidden = torch.cat([full_hidden, hidden_squeezed], dim=0)

    # Now full_hidden has shape: [prompt_len + num_generated, d_model]

    full_hidden = full_hidden.to("cuda")
    sae_acts = sae.encode(full_hidden.to(torch.float32))  # [total_len, num_features]

    # Get AVERAGE activations across entire sequence
    mean_acts = sae_acts.mean(0)  # [num_features]

    if use_tfidf_filtering:
        if np_client is None:
            raise ValueError("np_client must be provided when use_tfidf_filtering=True")

        print(f"\n--- Applying TF-IDF (rarity-weighted) filtering ---")

        # Get ALL active features (not just top k)
        min_activation = 0.0
        active_mask = mean_acts > min_activation
        active_indices = torch.where(active_mask)[0]

        print(f"  Found {len(active_indices)} active features")
        print(f"  Fetching Neuronpedia data and filtering by density < {max_density}...")

        features_info = []
        filtered_count = 0

        for i, feat_idx in enumerate(active_indices):
            feat_idx_int = feat_idx.item()
            activation = mean_acts[feat_idx].item()

            # Rate limiting: add delay between API calls
            if i > 0 and api_delay > 0:
                time.sleep(api_delay)

            # Fetch from Neuronpedia
            np_info = np_client.get_feature(feat_idx_int)

            # Filter by density
            if np_info.frac_nonzero is not None and np_info.frac_nonzero > max_density:
                filtered_count += 1
                continue

            # Compute activation-weighted rarity score
            rarity = compute_rarity_from_density(np_info.frac_nonzero)
            weighted_score = activation * (rarity ** rarity_power)

            features_info.append({
                'feature_idx': feat_idx_int,
                'activation': activation,
                'frac_nonzero': np_info.frac_nonzero,
                'rarity': rarity,
                'weighted_score': weighted_score,
                'description': np_info.description,
                'dashboard_url': np_client.get_dashboard_url(feat_idx_int)
            })

            # Progress indicator for long fetches
            if (i + 1) % 50 == 0:
                print(f"    Processed {i + 1}/{len(active_indices)} features...")

        print(f"  Filtered out {filtered_count} high-density features")
        print(f"  Remaining: {len(features_info)} features")

        # Sort by weighted score and return top-k
        features_info.sort(key=lambda x: x['weighted_score'], reverse=True)
        features_info = features_info[:k]

        # Extract top features from filtered results
        top_latents = torch.tensor([f['feature_idx'] for f in features_info])
        top_acts = torch.tensor([f['activation'] for f in features_info])

        print("\nTop features by activation-weighted rarity (prompt + output):")
        print("-" * 80)
        for i, info in enumerate(features_info):
            desc = info['description'] if info['description'] else "(No description)"
            if len(desc) > 60:
                desc = desc[:57] + "..."
            density_str = f"{info['frac_nonzero']:.5f}" if info['frac_nonzero'] else "N/A"
            print(f"{i+1:2d}. [{info['feature_idx']:6d}] Score: {info['weighted_score']:8.1f} | "
                  f"Act: {info['activation']:6.2f} | density={density_str} | {desc}")

        return top_acts, top_latents, sae_acts, outputs[0], generated_text, features_info

    else:
        # Standard approach: just take top k by average activation
        top_acts, top_latents = mean_acts.topk(k)

        print("Top features across FULL generation (prompt + output):")
        print("-" * 50)
        for act, idx in zip(top_acts, top_latents):
            print(f"{act:>6.1f} | Feature {idx}")

        return top_acts, top_latents, sae_acts, outputs[0], generated_text, None

In [15]:
# Initialize Neuronpedia client
np_client = NeuronpediaClient(
    model_id="gemma-3-4b-it",
    sae_id="29-gemmascope-2-mlp-262k"
)

# Run with TF-IDF filtering
LAYER = 29
top_acts, top_latents, sae_acts, token_ids, generated_text, features_info = get_features_with_full_generation(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    sample_input=formatted_prompt,
    layer_idx=LAYER,
    k=10,
    max_new_tokens=500,
    # TF-IDF filtering parameters
    use_tfidf_filterin
    g=True,
    np_client=np_client,
    max_density=0.005,  # Filter out features with >0.5% density : Borrowed from Daniel's code
    rarity_power=2.0,   # Square the rarity to weight it more heavily
    api_delay=0.1
)

print("\n" + "=" * 80)
print("FULL GENERATED OUTPUT:")
print("=" * 80)
print(generated_text)
print("=" * 80)

# Visualize the top feature
if features_info:
    print(f"\nVisualizing Feature {top_latents[0].item()} across ENTIRE generation:")
    print(f"Description: {features_info[0]['description']}")
    print("(Darker red = higher activation)\n")
    visualize_feature_on_full_generation(tokenizer, token_ids, sae_acts, top_latents[0].item())

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Applying TF-IDF (rarity-weighted) filtering ---
  Found 2703 active features
  Fetching Neuronpedia data and filtering by density < 0.005...
    Processed 50/2703 features...
    Processed 100/2703 features...
    Processed 150/2703 features...
    Processed 200/2703 features...
    Processed 250/2703 features...
    Processed 300/2703 features...
    Processed 350/2703 features...
    Processed 400/2703 features...
    Processed 450/2703 features...
    Processed 500/2703 features...
    Processed 550/2703 features...
    Processed 600/2703 features...
    Processed 650/2703 features...
    Processed 700/2703 features...
    Processed 750/2703 features...
    Processed 800/2703 features...
    Processed 850/2703 features...
    Processed 900/2703 features...
    Processed 950/2703 features...
    Processed 1000/2703 features...
    Processed 1050/2703 features...
    Processed 1100/2703 features...
    Processed 1150/2703 features...
    Processed 1200/2703 features...
    Proces

NameError: name 'visualize_feature_on_full_generation' is not defined

In [17]:
# Visualize the top feature
if features_info:
    print(f"\nVisualizing Feature {top_latents[0].item()} across ENTIRE generation:")
    print(f"Description: {features_info[0]['description']}")
    print("(Darker red = higher activation)\n")
    visualize_feature_on_full_generation(tokenizer, token_ids, sae_acts, top_latents[0].item())


Visualizing Feature 5149 across ENTIRE generation:
Description: None
(Darker red = higher activation)



In [45]:
def analyze_prompt_vs_generation(tokenizer, token_ids, sae_acts, feature_idx, prompt_text):
    """
    Compare how a feature activates in prompt vs generation regions.
    """

    # Find where prompt ends
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
    prompt_len = len(prompt_ids)

    activations = sae_acts[:, feature_idx].cpu().detach().numpy()

    prompt_acts = activations[:prompt_len]
    gen_acts = activations[prompt_len:]

    print(f"\nFeature {feature_idx} Analysis:")
    print("-" * 50)
    print(f"Prompt region ({prompt_len} tokens):")
    print(f"  Mean: {prompt_acts.mean():.2f}, Max: {prompt_acts.max():.2f}")
    print(f"\nGeneration region ({len(gen_acts)} tokens):")
    print(f"  Mean: {gen_acts.mean():.2f}, Max: {gen_acts.max():.2f}")

    # Show where it activates most
    top_5_idx = np.argsort(activations)[-5:][::-1]
    print(f"\nTop 5 activating positions:")
    for i, idx in enumerate(top_5_idx, 1):
        token = tokenizer.decode([token_ids[idx]])
        region = "PROMPT" if idx < prompt_len else "GENERATED"
        print(f"  {i}. Pos {idx} ({region}): '{token}' → {activations[idx]:.2f}")

# Use it
analyze_prompt_vs_generation(
    tokenizer=tokenizer,
    token_ids=token_ids,
    sae_acts=sae_acts,
    feature_idx=top_latents[0].item(),
    prompt_text=formatted_prompt
)


Feature 7290 Analysis:
--------------------------------------------------
Prompt region (118 tokens):
  Mean: 123.29, Max: 1579.55

Generation region (231 tokens):
  Mean: 290.31, Max: 1724.32

Top 5 activating positions:
  1. Pos 257 (GENERATED): 'Most' → 1724.32
  2. Pos 258 (GENERATED): ' job' → 1680.07
  3. Pos 63 (PROMPT): ' during' → 1579.55
  4. Pos 291 (GENERATED): ' most' → 1558.37
  5. Pos 310 (GENERATED): ' most' → 1501.57


In [72]:
def visualize_feature_on_full_generation_with_line_sums(tokenizer, token_ids, sae_acts, feature_idx):
    """
    Visualize a feature's activations across the entire generation.
    Red intensity shows activation strength.
    Preserves original text formatting (newlines, spacing).
    Also computes and displays sum of activations for each line.
    """

    # Decode each token individually to get token boundaries
    str_tokens = [tokenizer.decode([tok_id]) for tok_id in token_ids]

    # Get activations for this feature
    activations = sae_acts[:, feature_idx].cpu().detach().numpy()

    # Normalize to 0-1 for color intensity
    normalized = activations / (1e-6 + np.max(activations))

    # Track lines and their activations
    current_line_tokens = []
    current_line_acts = []
    lines_data = []  # List of (line_text, activation_sum, html)

    # Create HTML with colored spans, preserving original text structure
    current_line_html = []

    for i, (token, intensity, act_value) in enumerate(zip(str_tokens, normalized, activations)):
        # Check if this token contains a newline
        has_newline = '\n' in token

        # Add token to current line
        current_line_tokens.append(token)
        current_line_acts.append(act_value)

        # Escape HTML special characters
        token_display = (token
                        .replace('&', '&amp;')
                        .replace('<', '&lt;')
                        .replace('>', '&gt;')
                        .replace('\n', '<br>')  # Preserve line breaks
                        .replace(' ', '&nbsp;'))  # Preserve spaces

        span = (f'<span style="background-color: rgba(255,0,0,{intensity}); '
                f'color: black; padding: 2px 0px; margin: 0;">{token_display}</span>')

        current_line_html.append(span)

        # If we hit a newline, save the current line
        if has_newline:
            line_text = ''.join(current_line_tokens).strip()
            line_sum = sum(current_line_acts)
            line_html = ''.join(current_line_html)

            lines_data.append({
                'text': line_text,
                'sum': line_sum,
                'html': line_html,
                'num_tokens': len(current_line_tokens)
            })

            # Reset for next line
            current_line_tokens = []
            current_line_acts = []
            current_line_html = []

    # Don't forget the last line if it doesn't end with newline
    if current_line_tokens:
        line_text = ''.join(current_line_tokens).strip()
        line_sum = sum(current_line_acts)
        line_html = ''.join(current_line_html)

        lines_data.append({
            'text': line_text,
            'sum': line_sum,
            'html': line_html,
            'num_tokens': len(current_line_tokens)
        })

    # Create the full HTML output with line numbers and sums
    full_html_parts = []

    for i, line_data in enumerate(lines_data, 1):
        # Add line with its activation sum
        full_html_parts.append(
            f'<div style="margin-bottom: 5px; border-left: 3px solid #ddd; padding-left: 10px;">'
            f'<div style="font-size: 10px; color: #666; margin-bottom: 2px;">'
            f'Line {i} | Activation Sum: {line_data["sum"]:.2f} | Tokens: {line_data["num_tokens"]}'
            f'</div>'
            f'<div>{line_data["html"]}</div>'
            f'</div>'
        )

    full_html = ''.join(full_html_parts)

    # Display with pre-formatted text styling
    display(HTML(f'''
        <div style="font-family: 'Courier New', monospace;
                    font-size: 13px;
                    padding: 15px;
                    background: white;
                    border: 1px solid #ddd;
                    border-radius: 5px;
                    overflow-x: auto;
                    line-height: 1.6;">
            {full_html}
        </div>
    '''))

    # Also print a summary
    print("\n" + "=" * 80)
    print(f"LINE-BY-LINE ACTIVATION SUMMARY (Feature {feature_idx})")
    print("=" * 80)

    for i, line_data in enumerate(lines_data, 1):
        # Truncate long lines for display
        display_text = line_data['text'][:60] + '...' if len(line_data['text']) > 60 else line_data['text']
        print(f"Line {i:3d}: Sum={line_data['sum']:>8.2f} | Tokens={line_data['num_tokens']:>3d} | {display_text}")

    return lines_data


# Alternative: Simpler version that just prints the sums without modifying visualization
def print_line_activation_sums(tokenizer, token_ids, sae_acts, feature_idx):
    """
    Just compute and print activation sums per line without changing visualization.
    """

    str_tokens = [tokenizer.decode([tok_id]) for tok_id in token_ids]
    activations = sae_acts[:, feature_idx].cpu().detach().numpy()

    # Track lines
    lines = []
    current_line_tokens = []
    current_line_acts = []

    for token, act_value in zip(str_tokens, activations):
        current_line_tokens.append(token)
        current_line_acts.append(act_value)

        if '\n' in token:
            line_text = ''.join(current_line_tokens).strip()
            line_sum = sum(current_line_acts)
            lines.append({
                'text': line_text,
                'sum': line_sum,
                'num_tokens': len(current_line_tokens)
            })
            current_line_tokens = []
            current_line_acts = []

    # Last line
    if current_line_tokens:
        line_text = ''.join(current_line_tokens).strip()
        line_sum = sum(current_line_acts)
        lines.append({
            'text': line_text,
            'sum': line_sum,
            'num_tokens': len(current_line_tokens)
        })

    print("\n" + "=" * 80)
    print(f"LINE-BY-LINE ACTIVATION SUMS (Feature {feature_idx})")
    print("=" * 80)
    print(f"{'Line':<6} {'Sum':>10} {'Tokens':>8} {'Text Preview'}")
    print("-" * 80)

    for i, line_data in enumerate(lines, 1):
        display_text = line_data['text'][:50] + '...' if len(line_data['text']) > 50 else line_data['text']
        print(f"{i:<6} {line_data['sum']:>10.2f} {line_data['num_tokens']:>8} {display_text}")

    return lines


# Usage:

# Option 1: Visualize with line sums embedded
lines_data = visualize_feature_on_full_generation_with_line_sums(
    tokenizer=tokenizer,
    token_ids=token_ids,
    sae_acts=sae_acts,
    feature_idx=top_latents[0].item()
)

# Option 2: Keep original visualization, just print sums separately
visualize_feature_on_full_generation(tokenizer, token_ids, sae_acts, top_latents[0].item())
lines = print_line_activation_sums(tokenizer, token_ids, sae_acts, top_latents[0].item())


LINE-BY-LINE ACTIVATION SUMMARY (Feature 6196)
Line   1: Sum= 1407.57 | Tokens= 19 | <bos><start_of_turn>user Question: As the fox ran into the f...
Line   2: Sum= 5378.99 | Tokens= 23 | Options: a) nantucket b) barn c) northern hemisphere d) hen ...
Line   3: Sum= 9500.65 | Tokens= 15 | Before selecting an answer, explain your reasoning process. ...
Line   4: Sum= 2430.68 | Tokens= 13 | - What is the typical emotional state during job application...
Line   5: Sum= 2499.59 | Tokens= 11 | - Which option best captures this emotional state?
Line   6: Sum= 7021.92 | Tokens= 11 | - Why are the other options less suitable?
Line   7: Sum= 3056.83 | Tokens= 14 | Reasoning: [Your step-by-step thinking]
Line   8: Sum= 6466.73 | Tokens= 11 | Answer: [Your final choice] <end_of_turn>
Line   9: Sum= 3497.48 | Tokens= 14 | <start_of_turn>modelOkay, let’s break this riddle down!
Line  10: Sum= 1641.57 | Tokens=  5 | **Reasoning:**
Line  11: Sum= 3974.73 | Tokens= 32 | The riddle is a classic word pu


LINE-BY-LINE ACTIVATION SUMS (Feature 6196)
Line          Sum   Tokens Text Preview
--------------------------------------------------------------------------------
1         1407.57       19 <bos><start_of_turn>user Question: As the fox ran ...
2         5378.99       23 Options: a) nantucket b) barn c) northern hemisphe...
3         9500.65       15 Before selecting an answer, explain your reasoning...
4         2430.68       13 - What is the typical emotional state during job a...
5         2499.59       11 - Which option best captures this emotional state?
6         7021.92       11 - Why are the other options less suitable?
7         3056.83       14 Reasoning: [Your step-by-step thinking]
8         6466.73       11 Answer: [Your final choice] <end_of_turn>
9         3497.48       14 <start_of_turn>modelOkay, let’s break this riddle ...
10        1641.57        5 **Reasoning:**
11        3974.73       32 The riddle is a classic word puzzle relying on a b...
12        1083.58     

In [70]:
def print_option_lines_activation_sums(tokenizer, token_ids, sae_acts, feature_idx):
    """
    Compute and print activation sums only for lines containing options (a, b, c, d, e).
    """

    str_tokens = [tokenizer.decode([tok_id]) for tok_id in token_ids]
    activations = sae_acts[:, feature_idx].cpu().detach().numpy()

    # Track lines
    lines = []
    current_line_tokens = []
    current_line_acts = []

    for token, act_value in zip(str_tokens, activations):
        current_line_tokens.append(token)
        current_line_acts.append(act_value)

        if '\n' in token:
            line_text = ''.join(current_line_tokens).strip()
            line_sum = sum(current_line_acts)
            lines.append({
                'text': line_text,
                'sum': line_sum,
                'num_tokens': len(current_line_tokens)
            })
            current_line_tokens = []
            current_line_acts = []

    # Last line
    if current_line_tokens:
        line_text = ''.join(current_line_tokens).strip()
        line_sum = sum(current_line_acts)
        lines.append({
            'text': line_text,
            'sum': line_sum,
            'num_tokens': len(current_line_tokens)
        })

    # Filter for lines containing options
    option_lines = []
    for i, line_data in enumerate(lines, 1):
        line_text = line_data['text'].lower()
        # Check if line contains option markers
        if any(f'{opt})' in line_text for opt in ['a', 'b', 'c', 'd', 'e']):
            option_lines.append((i, line_data))

    if not option_lines:
        print("No option lines found!")
        return []

    print("\n" + "=" * 80)
    print(f"OPTIONS LINE ACTIVATION SUMS (Feature {feature_idx})")
    print("=" * 80)
    print(f"{'Line':<6} {'Sum':>10} {'Tokens':>8} {'Text'}")
    print("-" * 80)

    for line_num, line_data in option_lines:
        print(f"{line_num:<6} {line_data['sum']:>10.2f} {line_data['num_tokens']:>8} {line_data['text']}")

    return option_lines


# Alternative: Extract individual options from the options line
def analyze_individual_options_from_line(tokenizer, token_ids, sae_acts, feature_idx):
    """
    Find the line with options and compute activation sums for each individual option.
    """

    str_tokens = [tokenizer.decode([tok_id]) for tok_id in token_ids]
    activations = sae_acts[:, feature_idx].cpu().detach().numpy()

    # Find the line containing "Options:"
    current_pos = 0
    options_line_start = None
    options_line_end = None

    full_text = ""
    for i, token in enumerate(str_tokens):
        prev_text = full_text
        full_text += token

        # Detect start of options line
        if "Options:" in full_text and "Options:" not in prev_text:
            options_line_start = i

        # Detect end of options line (newline after options started)
        if options_line_start is not None and '\n' in token:
            options_line_end = i + 1
            break

    if options_line_start is None:
        print("Could not find Options line!")
        return {}

    # Now parse individual options within this line
    results = {}
    option_markers = {}

    # Find where each option marker appears
    partial_text = ""
    for i in range(options_line_start, options_line_end):
        token = str_tokens[i]
        prev_partial = partial_text
        partial_text += token

        for opt in ['a', 'b', 'c', 'd', 'e']:
            marker = f"{opt})"
            if marker in partial_text and marker not in prev_partial:
                option_markers[opt] = i

    # Compute activation sums for each option
    option_letters = ['a', 'b', 'c', 'd', 'e']

    for i, opt in enumerate(option_letters):
        if opt not in option_markers:
            continue

        start_idx = option_markers[opt] + 1  # After the ')' token

        # End is before next option or end of line
        if i + 1 < len(option_letters) and option_letters[i + 1] in option_markers:
            end_idx = option_markers[option_letters[i + 1]] - 1  # Before next option's letter
        else:
            end_idx = options_line_end

        option_acts = activations[start_idx:end_idx]
        option_tokens = [str_tokens[j] for j in range(start_idx, end_idx)]
        option_text = ''.join(option_tokens).strip()

        results[opt] = {
            'sum': float(option_acts.sum()),
            'mean': float(option_acts.mean()) if len(option_acts) > 0 else 0.0,
            'num_tokens': len(option_acts),
            'text': option_text
        }

    print("\n" + "=" * 80)
    print(f"INDIVIDUAL OPTION ACTIVATION SUMS (Feature {feature_idx})")
    print("=" * 80)

    for opt in ['a', 'b', 'c', 'd', 'e']:
        if opt in results:
            data = results[opt]
            print(f"Option {opt}): {data['sum']:>8.2f} ({data['num_tokens']} tokens) - {data['text']}")

    return results


# Usage:

# # Option 1: Just print the options line(s)
# option_lines = print_option_lines_activation_sums(
#     tokenizer=tokenizer,
#     token_ids=token_ids,
#     sae_acts=sae_acts,
#     feature_idx=top_latents[0].item()
# )

# Option 2: Parse individual options from the options line
option_results = analyze_individual_options_from_line(
    tokenizer=tokenizer,
    token_ids=token_ids,
    sae_acts=sae_acts,
    feature_idx=top_latents[0].item())


INDIVIDUAL OPTION ACTIVATION SUMS (Feature 6196)
Option a):     0.00 (2 tokens) - bath tub
Option b):     0.00 (1 tokens) - finger
Option c):   311.78 (1 tokens) - windowsill
Option d):   200.74 (2 tokens) - wedding ceremony
Option e):  1456.80 (3 tokens) - a coma
